In [1]:
# debug_sphere.ipynb
import numpy as np
import sys, os
sys.path.insert(0, os.getcwd())

from src.geometry import Sphere, SphericalCap
from src.geometry import sphere_sphere_surface_distance
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity, particle_surface_distance

L = 10000.0
r = 200.0
thresh = 1.8
half = L/2

print("="*70)
print("第一步：验证 split_sphere_by_box 对跨越边界球的切割")
print("="*70)

s = Sphere(np.array([4900., 0., 0.]), r, id=0)
parts = split_sphere_by_box(s, L)

print(f"原始球: center={s.c}, r={s.r}")
print(f"切割后片段数: {len(parts)}")
for i, p in enumerate(parts):
    print(f"  片段{i}: {type(p).__name__}")
    if hasattr(p, 'c'):
        print(f"    center={p.c}")
    if hasattr(p, 'n') and hasattr(p, 'd'):
        print(f"    n={p.n}, d={p.d}")

print("\n" + "="*70)
print("第二步：验证粒子间距离计算（用分割后的片段）")
print("="*70)

if len(parts) >= 2:
    # 同一原始球的两个片段之间的距离
    dist_12 = particle_surface_distance(parts[0], parts[1])
    print(f"片段0 与 片段1 的粒子间距离: {dist_12:.4f} nm")
    print(f"  阈值: {thresh} nm, 是否连通: {dist_12 <= thresh}")
    print("  物理上同一原始球的两个部分相距一个周期，不应因位置接近而连通")

print("\n" + "="*70)
print("第三步：验证距离计算函数是否会被某个 '默认保守估计' 分支阻断")
print("="*70)

# 测试各种组合
s1 = Sphere(np.array([0., 0., 0.]), r, id=1)
s2 = Sphere(np.array([400., 0., 0.]), r, id=2)
# 这两个球表面距离为0，应连通
dist_ss = particle_surface_distance(s1, s2)
print(f"两个完整球 (中心距400): 距离={dist_ss:.4f}, 是否连通={dist_ss <= thresh}")

# 球冠与完整球
if len(parts) >= 1:
    dist_cap_sphere = particle_surface_distance(parts[0], s1)
    print(f"球冠与完整球: 距离={dist_cap_sphere:.4f}, 是否连通={dist_cap_sphere <= thresh}")

# 两个球冠
if len(parts) >= 2:
    dist_cap_cap = particle_surface_distance(parts[0], parts[1])
    print(f"两个球冠: 距离={dist_cap_cap:.4f}, 是否连通={dist_cap_cap <= thresh}")

print("\n" + "="*70)
print("第四步：用一排球（从左到右）测试 build_connectivity 是否正常工作")
print("="*70)

# 生成从左到右的一排球，确保接触电极
n_spheres = 35
x_positions = np.linspace(-4800, 4800, n_spheres)
spheres_chain = []
for i, x in enumerate(x_positions):
    spheres_chain.append(Sphere(np.array([x, 0., 0.]), r, id=i))

# 分割（完全在盒内，应该每个球返回1个片段）
segments_chain = []
for s in spheres_chain:
    segments_chain.extend(split_sphere_by_box(s, L))

print(f"生成 {len(spheres_chain)} 个球，分割后 {len(segments_chain)} 个片段")
print(f"  预期: {len(spheres_chain)} 个片段")

conn, uf = build_connectivity(segments_chain, L, thresh)
print(f"build_connectivity 结果: {conn}")

if not conn:
    print("\n🔍 调试: 检查电极接触和根节点")
    LEFT_NODE = ("PLANE", "LEFT")
    RIGHT_NODE = ("PLANE", "RIGHT")
    left_root = uf.find(LEFT_NODE)
    right_root = uf.find(RIGHT_NODE)
    print(f"左电极根: {left_root}")
    print(f"右电极根: {right_root}")
    print(f"两者是否相同: {left_root == right_root}")
    
    # 检查每个片段归属
    for i, seg in enumerate(segments_chain[:5]):  # 只打印前5个
        print(f"  片段{i}: 根={uf.find(i)}, center={seg.c}")

print("\n" + "="*70)
print("第五步：测试电极接触判定是否被意外过滤")
print("="*70)

# 单个球接触左电极
s_left = Sphere(np.array([-4800., 0., 0.]), r, id=99)
seg_left = split_sphere_by_box(s_left, L)
print(f"左电极球: center={s_left.c}, r={s_left.r}")
print(f"  左表面 = {s_left.c[0] - s_left.r} = {-4800-200} = {-5000}")
print(f"  是否接触左电极: 是")

conn_left, uf_left = build_connectivity(seg_left, L, thresh)
LEFT_NODE = ("PLANE", "LEFT")
print(f"单个左电极球: build_connectivity 结果 = {conn_left}")
print(f"  预期: False (因为没有右电极连接)")

# 检查该球是否真的连接到了左电极
left_root = uf_left.find(LEFT_NODE)
print(f"  左电极根: {left_root}")
for i, seg in enumerate(seg_left):
    print(f"  片段{i}: 根={uf_left.find(i)}, center={seg.c}")
    if uf_left.find(i) == left_root:
        print(f"    ✅ 片段{i} 连接到左电极")

第一步：验证 split_sphere_by_box 对跨越边界球的切割
原始球: center=[4900.    0.    0.], r=200.0
切割后片段数: 2
  片段0: SphericalCap
    center=[4900.    0.    0.]
    n=[-1.  0.  0.], d=-100.0
  片段1: SphericalCap
    center=[-5100.     0.     0.]
    n=[1. 0. 0.], d=100.0

第二步：验证粒子间距离计算（用分割后的片段）
片段0 与 片段1 的粒子间距离: 9600.0000 nm
  阈值: 1.8 nm, 是否连通: False
  物理上同一原始球的两个部分相距一个周期，不应因位置接近而连通

第三步：验证距离计算函数是否会被某个 '默认保守估计' 分支阻断
两个完整球 (中心距400): 距离=0.0000, 是否连通=True
球冠与完整球: 距离=4500.0000, 是否连通=False
两个球冠: 距离=9600.0000, 是否连通=False

第四步：用一排球（从左到右）测试 build_connectivity 是否正常工作
生成 35 个球，分割后 35 个片段
  预期: 35 个片段
build_connectivity 结果: True

第五步：测试电极接触判定是否被意外过滤
左电极球: center=[-4800.     0.     0.], r=200.0
  左表面 = -5000.0 = -5000 = -5000
  是否接触左电极: 是
单个左电极球: build_connectivity 结果 = False
  预期: False (因为没有右电极连接)
  左电极根: 0
  片段0: 根=0, center=[-4800.     0.     0.]
    ✅ 片段0 连接到左电极


In [1]:
import numpy as np
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity
from src.monte_carlo import sample_random_spheres

L = 10000.0
r = 200.0
thresh = 1.8

# ============================================================
# 测试1：人为紧密排列的球（确定导通）
# ============================================================
print("="*70)
print("测试1：人为紧密排列的球（球心距=300nm，确保连通）")
print("="*70)

# 沿 X 轴排列，间距 300nm，覆盖从 -4800 到 4800
x_positions = np.arange(-4800, 4801, 300)  # 33个点
spheres = [Sphere(np.array([x, 0., 0.]), r, id=i) for i, x in enumerate(x_positions)]

segments = []
for s in spheres:
    segments.extend(split_sphere_by_box(s, L))

print(f"球数: {len(spheres)}, 片段数: {len(segments)}")
connected, uf = build_connectivity(segments, L, thresh)
print(f"导通: {connected}")
print()

# ============================================================
# 测试2：随机分布 500 个球（体积分数约 1.67%）
# ============================================================
print("="*70)
print("测试2：随机分布 500 个球（f ≈ 1.67%）")
print("="*70)

N = 500
rng = np.random.default_rng(42)
centers = rng.uniform(-L/2, L/2, size=(N, 3))
spheres_rand = [Sphere(centers[i], r, id=i) for i in range(N)]

segments_rand = []
for s in spheres_rand:
    segs = split_sphere_by_box(s, L)
    segments_rand.extend(segs)

print(f"球数: {N}, 片段数: {len(segments_rand)}")
connected, uf = build_connectivity(segments_rand, L, thresh)
print(f"导通: {connected}")
print()

# ============================================================
# 测试3：随机分布 2000 个球（体积分数约 6.7%）
# ============================================================
print("="*70)
print("测试3：随机分布 2000 个球（f ≈ 6.7%）")
print("="*70)

N2 = 2000
rng2 = np.random.default_rng(123)
centers2 = rng2.uniform(-L/2, L/2, size=(N2, 3))
spheres_rand2 = [Sphere(centers2[i], r, id=i) for i in range(N2)]

segments_rand2 = []
for s in spheres_rand2:
    segs = split_sphere_by_box(s, L)
    segments_rand2.extend(segs)

print(f"球数: {N2}, 片段数: {len(segments_rand2)}")
connected2, uf2 = build_connectivity(segments_rand2, L, thresh)
print(f"导通: {connected2}")
print()

# ============================================================
# 测试4：随机分布 5000 个球（体积分数约 16.7%）
# ============================================================
print("="*70)
print("测试4：随机分布 5000 个球（f ≈ 16.7%）")
print("="*70)

N3 = 5000
rng3 = np.random.default_rng(456)
centers3 = rng3.uniform(-L/2, L/2, size=(N3, 3))
spheres_rand3 = [Sphere(centers3[i], r, id=i) for i in range(N3)]

segments_rand3 = []
for s in spheres_rand3:
    segs = split_sphere_by_box(s, L)
    segments_rand3.extend(segs)

print(f"球数: {N3}, 片段数: {len(segments_rand3)}")
connected3, uf3 = build_connectivity(segments_rand3, L, thresh)
print(f"导通: {connected3}")

测试1：人为紧密排列的球（球心距=300nm，确保连通）
球数: 33, 片段数: 33
导通: True

测试2：随机分布 500 个球（f ≈ 1.67%）
球数: 500, 片段数: 565
导通: False

测试3：随机分布 2000 个球（f ≈ 6.7%）
球数: 2000, 片段数: 2245
导通: False

测试4：随机分布 5000 个球（f ≈ 16.7%）
球数: 5000, 片段数: 5639
导通: False


In [4]:
N_high = 6000  # f ≈ 20%
spheres_high = sample_random_spheres(N_high, L, r, 42)  # 去掉 seed=，直接用位置参数
segments_high = []
for s in spheres_high:
    segments_high.extend(split_sphere_by_box(s, L))
connected_high, _ = build_connectivity(segments_high, L, thresh)
print(f"f≈20%, N={N_high}, 导通: {connected_high}")

f≈20%, N=6000, 导通: False


In [1]:
from src.geometry import Sphere, sphere_sphere_surface_distance
import numpy as np

# 两个球体，中心距 400nm（刚好接触，因为半径都是 200nm）
s1 = Sphere(np.array([0, 0, 0]), 200.0, id=0)
s2 = Sphere(np.array([400, 0, 0]), 200.0, id=1)

dist = sphere_sphere_surface_distance(s1, s2)
print(f"表面距离: {dist}")  # 应为 0.0

表面距离: 0.0


In [2]:
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
import numpy as np

s = Sphere(np.array([0, 0, 0]), 200.0, id=0)
segs = split_sphere_by_box(s, 10000.0)
print(f"截断后片段数: {len(segs)}")
for seg in segs:
    print(type(seg))

截断后片段数: 1
<class 'src.geometry.Sphere'>


In [3]:
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity
import numpy as np

L = 10000.0
thresh = 1.8

# 两个球体刚好接触（距离 400nm，表面距离 0）
s1 = Sphere(np.array([0, 0, 0]), 200.0, id=0)
s2 = Sphere(np.array([400, 0, 0]), 200.0, id=1)

segments = split_sphere_by_box(s1, L) + split_sphere_by_box(s2, L)
connected, uf = build_connectivity(segments, L, thresh)
print(f"两个接触球体导通: {connected}")  # 应为 True

两个接触球体导通: False


In [4]:
from src.connectivity import particle_surface_distance

dist = particle_surface_distance(s1, s2, mode='fast')
print(f"particle_surface_distance: {dist}")  # 应为 0.0

particle_surface_distance: 0.0


In [1]:
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity
import numpy as np

L = 10000.0
thresh = 1.8

s1 = Sphere(np.array([0, 0, 0]), 200.0, id=0)
s2 = Sphere(np.array([400, 0, 0]), 200.0, id=1)

segments = split_sphere_by_box(s1, L) + split_sphere_by_box(s2, L)
connected, uf = build_connectivity(segments, L, thresh)
print(f"两个接触球体导通: {connected}")  # 应为 True

DEBUG: i=0, j=1, dist=0.0, thresh=1.8
DEBUG: 合并 0 和 1
两个接触球体导通: False


In [2]:
s1 = Sphere(np.array([-5000, 0, 0]), 200.0, id=0)  # 接触左电极
s2 = Sphere(np.array([5000, 0, 0]), 200.0, id=1)   # 接触右电极

segments = split_sphere_by_box(s1, L) + split_sphere_by_box(s2, L)
connected, uf = build_connectivity(segments, L, thresh)
print(f"两个球体分别接触左右电极: {connected}")  # 应为 True

DEBUG: i=0, j=1, dist=9600.0, thresh=1.8
DEBUG: i=0, j=2, dist=0.0, thresh=1.8
DEBUG: 合并 0 和 2
DEBUG: i=0, j=3, dist=9600.0, thresh=1.8
DEBUG: i=1, j=2, dist=9600.0, thresh=1.8
DEBUG: i=1, j=3, dist=0.0, thresh=1.8
DEBUG: 合并 1 和 3
DEBUG: i=2, j=3, dist=9600.0, thresh=1.8
两个球体分别接触左右电极: False


In [4]:
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity
import numpy as np

L = 10000.0
thresh = 1.8

# 左电极球
s1 = Sphere(np.array([-5000, 0, 0]), 200.0, id=0)
# 中间球1
s2 = Sphere(np.array([-4600, 0, 0]), 200.0, id=1)
# 中间球2（与中间球1刚好接触，中心距 400nm）
s3 = Sphere(np.array([-4200, 0, 0]), 200.0, id=2)
# 右电极球
s4 = Sphere(np.array([5000, 0, 0]), 200.0, id=3)

# 检查：
# 左电极球(-5000) → 中间球1(-4600) 中心距400，表面距离0 → 导通
# 中间球1(-4600) → 中间球2(-4200) 中心距400，表面距离0 → 导通
# 中间球2(-4200) → 右电极球(5000) 中心距9200，表面距离8800 → 不导通！

# 修正：让中间球2也接触右电极
# 把右电极球移到 -3800（让中间球2和右电极球接触）
s4 = Sphere(np.array([-3800, 0, 0]), 200.0, id=3)
# 但这样右电极球就不在右电极位置了...

# 正确的测试：用一根长链连接左右电极
# 左电极球在 -5000，右电极球在 5000
# 中间放多个球，让它们依次接触

In [5]:
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity
import numpy as np

L = 10000.0
thresh = 1.8

# 创建一条从左到右的链
# 左电极 → 球1 → 球2 → 球3 → 右电极
# 每个球半径 200nm，中心距 400nm 时表面距离 0
spheres = [
    Sphere(np.array([-5000, 0, 0]), 200.0, id=0),  # 左电极
    Sphere(np.array([-4600, 0, 0]), 200.0, id=1),  # 与左电极接触
    Sphere(np.array([-4200, 0, 0]), 200.0, id=2),  # 与球1接触
    Sphere(np.array([-3800, 0, 0]), 200.0, id=3),  # 与球2接触
    # ... 继续延伸到右电极 5000，需要 5000 - (-3800) = 8800，需要 22 个球
]

# 或者更简单：直接用 3 个球
# 左电极球直接连到右电极球？不行，距离太远

# 最简单的方法：只用两个球，分别接触左右电极，然后它们之间也接触
# 左电极在 -5000，右电极在 5000，距离 10000
# 两个球半径 200，中心距 400 才能接触
# 所以需要一系列球：-5000, -4600, -4200, -3800, -3400, -3000, -2600, -2200, -1800, -1400, -1000, -600, -200, 200, 600, 1000, 1400, 1800, 2200, 2600, 3000, 3400, 3800, 4200, 4600, 5000
# 共 26 个球

# 但我们可以用更简单的方法：用 4 个球覆盖 5000 距离
# 每个球直径 400，4 个球最多覆盖 1600，不够 10000

In [6]:
import numpy as np
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity

L = 10000.0
thresh = 1.8
r = 200.0
N = 20  # 20 个球覆盖 10000 nm，间距约 526 nm（稍微大一点，需要 380 才行）
# 用 26 个球，间距 380 nm
N_spheres = 26
centers = np.linspace(-5000 + r, 5000 - r, N_spheres)
spheres = []
for i, c in enumerate(centers):
    spheres.append(Sphere(np.array([c, 0, 0]), r, id=i))

segments = []
for s in spheres:
    segments.extend(split_sphere_by_box(s, L))
connected, uf = build_connectivity(segments, L, thresh)
print(f"26球链导通: {connected}")  # 应该 True

DEBUG: i=0, j=1, dist=0.0, thresh=1.8
DEBUG: 合并 0 和 1
DEBUG: i=0, j=2, dist=368.0, thresh=1.8
DEBUG: i=0, j=3, dist=752.0, thresh=1.8
DEBUG: i=0, j=4, dist=1136.0, thresh=1.8
DEBUG: i=0, j=5, dist=1520.0, thresh=1.8
DEBUG: i=0, j=6, dist=1904.0, thresh=1.8
DEBUG: i=0, j=7, dist=2288.0, thresh=1.8
DEBUG: i=0, j=8, dist=2672.0, thresh=1.8
DEBUG: i=0, j=9, dist=3056.0, thresh=1.8
DEBUG: i=0, j=10, dist=3440.0, thresh=1.8
DEBUG: i=0, j=11, dist=3824.0, thresh=1.8
DEBUG: i=0, j=12, dist=4208.0, thresh=1.8
DEBUG: i=0, j=13, dist=4592.0, thresh=1.8
DEBUG: i=0, j=14, dist=4976.0, thresh=1.8
DEBUG: i=0, j=15, dist=5360.0, thresh=1.8
DEBUG: i=0, j=16, dist=5744.0, thresh=1.8
DEBUG: i=0, j=17, dist=6128.0, thresh=1.8
DEBUG: i=0, j=18, dist=6512.0, thresh=1.8
DEBUG: i=0, j=19, dist=6896.0, thresh=1.8
DEBUG: i=0, j=20, dist=7280.0, thresh=1.8
DEBUG: i=0, j=21, dist=7664.0, thresh=1.8
DEBUG: i=0, j=22, dist=8048.0, thresh=1.8
DEBUG: i=0, j=23, dist=8432.0, thresh=1.8
DEBUG: i=0, j=24, dist=8816.0, t

In [7]:
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity
import numpy as np

L = 10000.0
r = 200.0
thresh = 1.8
N = 209
seed = 20260812

rng = np.random.default_rng(seed)
centers = rng.uniform(-L/2, L/2, size=(N, 3))
spheres = [Sphere(c, r, id=i) for i, c in enumerate(centers)]

segments = []
for s in spheres:
    segs = split_sphere_by_box(s, L)
    segments.extend(segs)

print(f"原始球体: {len(spheres)}, 截断后: {len(segments)}")

# 检查是否有粒子接触电极
left_count = 0
right_count = 0
for s in segments:
    if isinstance(s, Sphere):
        if abs(s.c[0] + 5000) <= r + thresh:
            left_count += 1
        if abs(s.c[0] - 5000) <= r + thresh:
            right_count += 1
    # SphericalCap 的电极接触需要特殊处理
print(f"接触左电极的粒子数: {left_count}")
print(f"接触右电极的粒子数: {right_count}")

connected, uf = build_connectivity(segments, L, thresh)
print(f"连通性: {connected}")

原始球体: 209, 截断后: 238
接触左电极的粒子数: 0
接触右电极的粒子数: 0
DEBUG: i=0, j=1, dist=8795.263164277594, thresh=1.8
DEBUG: i=0, j=2, dist=8730.498864182433, thresh=1.8
DEBUG: i=0, j=3, dist=4401.754896657592, thresh=1.8
DEBUG: i=0, j=4, dist=8770.64945559717, thresh=1.8
DEBUG: i=0, j=5, dist=12074.600622882928, thresh=1.8
DEBUG: i=0, j=6, dist=7817.843369890903, thresh=1.8
DEBUG: i=0, j=7, dist=13322.380685064241, thresh=1.8
DEBUG: i=0, j=8, dist=5450.898454744446, thresh=1.8
DEBUG: i=0, j=9, dist=5904.065193728243, thresh=1.8
DEBUG: i=0, j=10, dist=7204.149904508925, thresh=1.8
DEBUG: i=0, j=11, dist=10635.278255677593, thresh=1.8
DEBUG: i=0, j=12, dist=9348.191509651631, thresh=1.8
DEBUG: i=0, j=13, dist=5176.319454949303, thresh=1.8
DEBUG: i=0, j=14, dist=3625.8399349083725, thresh=1.8
DEBUG: i=0, j=15, dist=10798.288482380533, thresh=1.8
DEBUG: i=0, j=16, dist=5498.694429551794, thresh=1.8
DEBUG: i=0, j=17, dist=8603.285043955158, thresh=1.8
DEBUG: i=0, j=18, dist=8472.928444155283, thresh=1.8
DEBUG

In [8]:
import numpy as np
from src.geometry import Sphere
from src.truncation import split_sphere_by_box

L = 10000.0
r = 200.0
N = 209
seed = 20260812

rng = np.random.default_rng(seed)
centers = rng.uniform(-L/2, L/2, size=(N, 3))

# 检查球心 X 坐标的分布
xs = centers[:, 0]
print(f"X 坐标范围: {xs.min():.1f} ~ {xs.max():.1f}")
print(f"在 [-4800, -5200] 的球数: {np.sum((xs >= -5200) & (xs <= -4800))}")
print(f"在 [4800, 5200] 的球数: {np.sum((xs >= 4800) & (xs <= 5200))}")

# 如果这些值都是 0，说明球心没有落在边界附近

X 坐标范围: -4973.5 ~ 4957.5
在 [-4800, -5200] 的球数: 2
在 [4800, 5200] 的球数: 5


In [10]:
from src.geometry import Sphere
import numpy as np

L = 10000.0
r = 200.0
thresh = 1.8
left_plane_x = -5000.0
right_plane_x = 5000.0

# 找到接触左电极的球体
for s in segments:
    if isinstance(s, Sphere):
        d_left = abs(s.c[0] - left_plane_x)
        surf_left = max(0.0, d_left - s.r)
        if surf_left <= thresh + 1e-9:
            print(f"球 {s.id} 接触左电极, 球心 x={s.c[0]}, surf_left={surf_left}")

In [12]:
from src.geometry import Sphere, SphericalCap
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity
import numpy as np

L = 10000.0
r = 200.0
thresh = 1.8
N = 209
seed = 20260812

rng = np.random.default_rng(seed)
centers = rng.uniform(-L/2, L/2, size=(N, 3))
spheres = [Sphere(c, r, id=i) for i, c in enumerate(centers)]

segments = []
for s in spheres:
    segs = split_sphere_by_box(s, L)
    segments.extend(segs)

print(f"原始球体: {len(spheres)}, 截断后粒子数: {len(segments)}")

# 检查电极接触
left_plane_x = -5000.0
right_plane_x = 5000.0

left_count = 0
right_count = 0

for i, s in enumerate(segments):
    if isinstance(s, Sphere):
        d_left = abs(s.c[0] - left_plane_x)
        surf_left = max(0.0, d_left - s.r)
        if surf_left <= thresh + 1e-9:
            left_count += 1
            print(f"球 {s.id} 接触左电极, 球心 x={s.c[0]:.2f}, surf_left={surf_left:.6f}")
        d_right = abs(s.c[0] - right_plane_x)
        surf_right = max(0.0, d_right - s.r)
        if surf_right <= thresh + 1e-9:
            right_count += 1
            print(f"球 {s.id} 接触右电极, 球心 x={s.c[0]:.2f}, surf_right={surf_right:.6f}")
    elif isinstance(s, SphericalCap):
        d_left = abs(s.c[0] - left_plane_x)
        surf_left = max(0.0, d_left - s.r)
        if surf_left <= thresh + 1e-9:
            left_count += 1
            print(f"球冠 {s.id} 接触左电极, 球心 x={s.c[0]:.2f}")
        d_right = abs(s.c[0] - right_plane_x)
        surf_right = max(0.0, d_right - s.r)
        if surf_right <= thresh + 1e-9:
            right_count += 1
            print(f"球冠 {s.id} 接触右电极, 球心 x={s.c[0]:.2f}")

print(f"接触左电极的粒子数: {left_count}")
print(f"接触右电极的粒子数: {right_count}")

connected, uf = build_connectivity(segments, L, thresh)
print(f"连通性: {connected}")

原始球体: 209, 截断后粒子数: 238
球冠 5 接触右电极, 球心 x=4886.27
球冠 5 接触左电极, 球心 x=-5113.73
球冠 34 接触左电极, 球心 x=-4973.49
球冠 34 接触右电极, 球心 x=5026.51
球冠 39 接触左电极, 球心 x=-4841.43
球冠 39 接触右电极, 球心 x=5158.57
球冠 92 接触右电极, 球心 x=4889.02
球冠 92 接触左电极, 球心 x=-5110.98
球冠 92 接触右电极, 球心 x=4889.02
球冠 92 接触右电极, 球心 x=4889.02
球冠 110 接触右电极, 球心 x=4957.48
球冠 110 接触左电极, 球心 x=-5042.52
球冠 123 接触右电极, 球心 x=4908.22
球冠 123 接触左电极, 球心 x=-5091.78
球冠 208 接触右电极, 球心 x=4885.10
球冠 208 接触左电极, 球心 x=-5114.90
接触左电极的粒子数: 7
接触右电极的粒子数: 9
DEBUG: i=0, j=1, dist=8795.263164277594, thresh=1.8
DEBUG: i=0, j=2, dist=8730.498864182433, thresh=1.8
DEBUG: i=0, j=3, dist=4401.754896657592, thresh=1.8
DEBUG: i=0, j=4, dist=8770.64945559717, thresh=1.8
DEBUG: i=0, j=5, dist=12074.600622882928, thresh=1.8
DEBUG: i=0, j=6, dist=7817.843369890903, thresh=1.8
DEBUG: i=0, j=7, dist=13322.380685064241, thresh=1.8
DEBUG: i=0, j=8, dist=5450.898454744446, thresh=1.8
DEBUG: i=0, j=9, dist=5904.065193728243, thresh=1.8
DEBUG: i=0, j=10, dist=7204.149904508925, thresh=1.8
DE

In [13]:
from src.geometry import Sphere, sphere_sphere_surface_distance
import numpy as np

# 两个球体中心距 400nm（表面距离应为 0）
s1 = Sphere(np.array([0, 0, 0]), 200.0, id=0)
s2 = Sphere(np.array([400, 0, 0]), 200.0, id=1)

dist = sphere_sphere_surface_distance(s1, s2)
print(f"表面距离: {dist}")  # 应为 0.0，如果不是 0，说明距离函数有问题

表面距离: 0.0


In [1]:
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity
import numpy as np

L = 10000.0
thresh = 1.8

s1 = Sphere(np.array([0, 0, 0]), 200.0, id=0)
s2 = Sphere(np.array([400, 0, 0]), 200.0, id=1)

segments = split_sphere_by_box(s1, L) + split_sphere_by_box(s2, L)
connected, uf = build_connectivity(segments, L, thresh)
print(f"两个接触球体导通: {connected}")  # 应为 True

DEBUG: i=0, j=1, dist=0.000000
DEBUG: 合并 0 和 1
两个接触球体导通: False


In [2]:
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity
import numpy as np

L = 10000.0
r = 200.0
thresh = 1.8
N = 209
seed = 20260812

# 直接随机生成球体
rng = np.random.default_rng(seed)
centers = rng.uniform(-L/2, L/2, size=(N, 3))
spheres = []
for i, c in enumerate(centers):
    spheres.append(Sphere(c, r, id=i))

# 截断
segments = []
for s in spheres:
    segs = split_sphere_by_box(s, L)
    segments.extend(segs)

print(f"原始球体: {len(spheres)}, 截断后粒子数: {len(segments)}")

# 检查电极接触
left_plane_x = -5000.0
right_plane_x = 5000.0
left_count = 0
right_count = 0
for s in segments:
    if isinstance(s, Sphere):
        if abs(s.c[0] - left_plane_x) <= r + thresh:
            left_count += 1
        if abs(s.c[0] - right_plane_x) <= r + thresh:
            right_count += 1
    elif hasattr(s, 'c'):  # SphericalCap
        if abs(s.c[0] - left_plane_x) <= r + thresh:
            left_count += 1
        if abs(s.c[0] - right_plane_x) <= r + thresh:
            right_count += 1
print(f"接触左电极: {left_count}, 接触右电极: {right_count}")

# 连通性判定
connected, uf = build_connectivity(segments, L, thresh)
print(f"连通性: {connected}")

原始球体: 209, 截断后粒子数: 238
接触左电极: 7, 接触右电极: 9
DEBUG: i=20, j=84, dist=123.524027
DEBUG: i=34, j=68, dist=105.253449
DEBUG: i=54, j=102, dist=0.000000
DEBUG: 合并 54 和 102
DEBUG: i=71, j=145, dist=196.291194
DEBUG: i=75, j=178, dist=124.184386
DEBUG: i=80, j=151, dist=230.182186
DEBUG: i=81, j=159, dist=0.000000
DEBUG: 合并 81 和 159
DEBUG: i=82, j=160, dist=0.000000
DEBUG: 合并 82 和 160
DEBUG: i=96, j=115, dist=35.810179
DEBUG: i=103, j=126, dist=0.000000
DEBUG: 合并 103 和 126
DEBUG: i=104, j=127, dist=0.000000
DEBUG: 合并 104 和 127
DEBUG: i=105, j=107, dist=0.000000
DEBUG: 合并 105 和 107
DEBUG: i=114, j=148, dist=108.715043
DEBUG: i=137, j=142, dist=16.687702
DEBUG: i=200, j=207, dist=0.000000
DEBUG: 合并 200 和 207
连通性: False


In [3]:
from src.geometry import Sphere
from src.truncation import split_sphere_by_box
from src.connectivity import build_connectivity
import numpy as np

L = 10000.0
r = 200.0
thresh = 1.8

# 25 个球从 x=-5000 到 x=5000，间距 400nm（刚好接触）
N = 25
spheres = []
for i in range(N):
    x = -5000 + r + i * 400
    spheres.append(Sphere(np.array([x, 0, 0]), r, id=i))

segments = []
for s in spheres:
    segs = split_sphere_by_box(s, L)
    segments.extend(segs)

connected, uf = build_connectivity(segments, L, thresh)
print(f"25球直线链导通: {connected}")

DEBUG: i=0, j=1, dist=0.000000
DEBUG: 合并 0 和 1
DEBUG: i=1, j=2, dist=0.000000
DEBUG: 合并 1 和 2
DEBUG: i=2, j=3, dist=0.000000
DEBUG: 合并 2 和 3
DEBUG: i=3, j=4, dist=0.000000
DEBUG: 合并 3 和 4
DEBUG: i=4, j=5, dist=0.000000
DEBUG: 合并 4 和 5
DEBUG: i=5, j=6, dist=0.000000
DEBUG: 合并 5 和 6
DEBUG: i=6, j=7, dist=0.000000
DEBUG: 合并 6 和 7
DEBUG: i=7, j=8, dist=0.000000
DEBUG: 合并 7 和 8
DEBUG: i=8, j=9, dist=0.000000
DEBUG: 合并 8 和 9
DEBUG: i=9, j=10, dist=0.000000
DEBUG: 合并 9 和 10
DEBUG: i=10, j=11, dist=0.000000
DEBUG: 合并 10 和 11
DEBUG: i=11, j=12, dist=0.000000
DEBUG: 合并 11 和 12
DEBUG: i=12, j=13, dist=0.000000
DEBUG: 合并 12 和 13
DEBUG: i=13, j=14, dist=0.000000
DEBUG: 合并 13 和 14
DEBUG: i=14, j=15, dist=0.000000
DEBUG: 合并 14 和 15
DEBUG: i=15, j=16, dist=0.000000
DEBUG: 合并 15 和 16
DEBUG: i=16, j=17, dist=0.000000
DEBUG: 合并 16 和 17
DEBUG: i=17, j=18, dist=0.000000
DEBUG: 合并 17 和 18
DEBUG: i=18, j=19, dist=0.000000
DEBUG: 合并 18 和 19
DEBUG: i=19, j=20, dist=0.000000
DEBUG: 合并 19 和 20
DEBUG: i=20, j=21,

In [4]:
sphere_count = 0
sphericalcap_count = 0
for s in segments:
    if isinstance(s, Sphere):
        sphere_count += 1
    elif hasattr(s, 'c') and hasattr(s, 'n'):  # SphericalCap
        sphericalcap_count += 1
print(f"Sphere: {sphere_count}, SphericalCap: {sphericalcap_count}")

Sphere: 25, SphericalCap: 0
